# CSV to FAIRFluids Conversion Example

This notebook demonstrates how to convert CSV data to FAIRFluids format using the updated `data_from_csv` method.

The `data_from_csv` method now:
- **Creates separate JSON documents** for each unique compound combination (Component#1, Component#2, Component#3)
- **Splits properties**: Each row creates separate fluids for viscosity and density
- **Single property per fluid**: Each fluid contains only one property type
- **Complete parameters**: Each fluid has mole fractions (X#1, X#2, X#3) and temperature as parameters

## Key Features
- **Compound combination grouping**: Each unique set of compounds gets its own document/JSON file
- **Property separation**: Viscosity and density from the same row create separate fluids in the same document
- **Automatic parameter creation**: Mole fractions and temperature are automatically extracted and added as parameters
- **Output to files**: Each compound combination is saved as a separate JSON file in the outputs folder


## Step 1: Import Required Libraries


In [ ]:
import sys
import os
import pandas as pd
import json
from pathlib import Path
# Add parent directory to path
sys.path.append('/home/YOURPATH/Code/FAIRFluids')

from fairfluids import FAIRFluidsDocument, Version, Citation, FluidIO


## Step 2: Load and Inspect the CSV Data


In [ ]:
# Path to the CSV file
csv_path = '/home/YOURPATH/Code/FAIRFluids/fairfluids/data/joined_viscosity_density.csv'

# Load the CSV
df = pd.read_csv(csv_path)

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("\nFirst few rows:")
df.head()


Total rows: 2159
Columns: ['Number of components', 'Type of DES', 'I', 'II', 'III', 'IV', 'V', 'Component#1', 'Component#2', 'Component#3', 'X#1 (molar fraction)', 'X#2 (molar fraction)', 'X#3 (molar fraction)', 'Temperature, K', 'Viscosity, cP', 'Reference (DOI)', 'Density, g/cm^3']

First few rows:


,Number of components,Type of DES,I,II,III,IV,V,Component#1,Component#2,Component#3,X#1 (molar fraction),X#2 (molar fraction),X#3 (molar fraction),"Temperature, K","Viscosity, cP",Reference (DOI),"Density, g/cm^3"
0,2,Ⅰ,1,0,0,0,0,zinc;dichloride,Benzyltributylammonium chloride,NaN,0.667,0.333,NaN,298.0,0.0135,10.13140/RG.2.2.13424.10245,1.00140
1,2,Ⅲ,0,0,1,0,0,1-Ethyl-3-methylimidazolium chloride,"1,2,4-TRIAZOLE",NaN,0.500,0.500,NaN,313.2,0.0347,10.1021/acssuschemeng.0c04215,1.16822
2,2,Ⅲ,0,0,1,0,0,1-Ethyl-3-methylimidazolium chloride,1H-Benzotriazole,NaN,0.500,0.500,NaN,313.2,0.0753,10.1021/acssuschemeng.0c04215,1.46660
3,2,Ⅲ,0,0,1,0,0,1-Ethyl-3-methylimidazolium chloride,1H-Tetrazole,NaN,0.500,0.500,NaN,313.2,0.0446,10.1021/acssuschemeng.0c04215,1.16062
4,2,Ⅲ,0,0,1,0,0,1-Ethyl-3-methylimidazolium chloride,BENZIMIDAZOLE,NaN,0.500,0.500,NaN,313.2,0.8254,10.1021/acssuschemeng.0c04215,1.19850


## Step 3: Use data_from_csv Directly

The updated `data_from_csv` method now reads the `joined_viscosity_density.csv` format directly!
No conversion needed. It will:
1. Group rows by unique compound combinations (Component#1, Component#2, Component#3)
2. Create a separate document for each combination
3. Split each row into viscosity and density fluids
4. Automatically extract parameters (mole fractions and temperature)
5. Save each document as a separate JSON file


In [12]:
# Show some example compound combinations
print("Example compound combinations from CSV:")
print(f"Total rows: {len(df)}")

# Show unique compound combinations
compound_combos = df[['Component#1', 'Component#2', 'Component#3']].drop_duplicates()
print(f"\nUnique compound combinations: {len(compound_combos)}")
print("\nFirst 10 combinations:")
compound_combos.head(10)


Example compound combinations from CSV:
Total rows: 2159

Unique compound combinations: 222

First 10 combinations:


,Component#1,Component#2,Component#3
0,zinc;dichloride,Benzyltributylammonium chloride,NaN
1,1-Ethyl-3-methylimidazolium chloride,"1,2,4-TRIAZOLE",NaN
2,1-Ethyl-3-methylimidazolium chloride,1H-Benzotriazole,NaN
3,1-Ethyl-3-methylimidazolium chloride,1H-Tetrazole,NaN
4,1-Ethyl-3-methylimidazolium chloride,BENZIMIDAZOLE,NaN
5,1-Ethyl-3-methylimidazolium chloride,imidazole,NaN
6,1-Ethyl-3-methylimidazolium chloride,phenol,NaN
7,1-Ethyl-3-methylimidazolium chloride,SUCCINIMIDE,NaN
8,"1-hydroxyethyl-1,4-dimethyl-piperazinium bromide",glycerol,NaN
29,ACETYLCHOLINE CHLORIDE,"1,2,4-TRIAZOLE",NaN


## Step 4: Create FAIRFluids Documents from CSV

Now we'll use the updated `data_from_csv` method to create FAIRFluids documents.
The method will:
- Group by unique compound combinations
- Create separate documents for each combination
- Split viscosity and density into separate fluids
- Save each document as a JSON file


In [13]:
# Create outputs directory if it doesn't exist
output_dir = Path('outputsss')
output_dir.mkdir(exist_ok=True)

# Create FluidIO instance
fluid_io = FluidIO()

# Create documents from CSV (one per compound combination)
# Set fetch_from_pubchem=True to fetch compound information from PubChem
print("Creating FAIRFluids documents from CSV...")
documents = fluid_io.data_from_csv(
    csv_path=csv_path,
    output_dir=str(output_dir),
    fetch_from_pubchem=True  # Set to True to fetch compound info from PubChem
)

print(f"\n✅ Created {len(documents)} documents (one per compound combination)")
print(f"✅ Saved JSON files to: {output_dir}")

Creating FAIRFluids documents from CSV...
Found 222 unique compound combinations

Fetching compound information from PubChem for 120 unique compounds...

Fetched data for 120 compounds

  Added compound: zinc;dichloride -> Zincchloride (CID: 3007855)
  Added compound: Benzyltributylammonium chloride -> Benzyltributylammonium Chloride (CID: 159952)
Saved document to: outputsss/zinc_dichloride_Benzyltributylammoni.json
  Added compound: 1-Ethyl-3-methylimidazolium chloride -> 1-Ethyl-3-Methylimidazolium Chloride (CID: 2734160)
  Added compound: 1,2,4-TRIAZOLE -> 1H-1,2,4-Triazole (CID: 9257)
Saved document to: outputsss/1-Ethyl-3-methylimid_124-TRIAZOLE.json
  Added compound: 1-Ethyl-3-methylimidazolium chloride -> 1-Ethyl-3-Methylimidazolium Chloride (CID: 2734160)
  Added compound: 1H-Benzotriazole -> 1H-Benzotriazole (CID: 7220)
Saved document to: outputsss/1-Ethyl-3-methylimid_1H-Benzotriazole.json
  Added compound: 1-Ethyl-3-methylimidazolium chloride -> 1-Ethyl-3-Methylimidazolium 

## Step 4.1
Load From Data

In [14]:
documents = []
for doc in Path('outputsss').glob('*.json'):
    documents.append(FAIRFluidsDocument.model_validate_json(doc.read_text()))

documents[0]

FAIRFluidsDocument(version=Version(versionMajor=1, versionMinor=0), citation=None, compound=[Compound(compoundID='L-Histidine', pubChemID=6274, commonName='L-Histidine', SELFIE=None, name_IUPAC='(2S)-2-amino-3-(1H-imidazol-5-yl)propanoic acid', standard_InChI='InChI=1S/C6H9N3O2/c7-5(6(10)11)1-4-2-8-3-9-4/h2-3,5H,1,7H2,(H,8,9)(H,10,11)/t5-/m0/s1', standard_InChI_key='HNDVDQJCIGZPNO-YFKPBYRVSA-N', molar_weigth=155.15, smiles_code='C1=C(NC=N1)C[C@@H](C(=O)O)N', sigma_profile=None), Compound(compoundID='Oxalic Acid', pubChemID=971, commonName='Oxalic Acid', SELFIE=None, name_IUPAC='oxalic acid', standard_InChI='InChI=1S/C2H2O4/c3-1(4)2(5)6/h(H,3,4)(H,5,6)', standard_InChI_key='MUBZPKHOEPUJKR-UHFFFAOYSA-N', molar_weigth=90.03, smiles_code='C(=O)(C(=O)O)O', sigma_profile=None)], fluid=[Fluid(fluidID=['9208ac1f-c861-4944-9ad5-813d2bf98c8b'], compounds=['L-Histidine', 'Oxalic Acid'], property=[Property(propertyID='viscosity', properties=<Properties.VISCOSITY: 'viscosity'>, unit=UnitDefinition(

## Step 5: Inspect the Created Documents

Let's examine one of the created documents to verify the structure.


In [15]:
# Inspect the first document
if documents:
    doc = documents[0]
    
    print(f"Document 1 summary:")
    print(f"  - Compounds: {doc.fluid[0].compounds if doc.fluid else 'None'}")
    print(f"  - Number of fluids: {len(doc.fluid)}")
    
    for i, fluid in enumerate(doc.fluid):
        print(f"\n  Fluid {i+1}:")
        print(f"    Compounds: {fluid.compounds}")
        print(f"    Properties: {[p.propertyID for p in fluid.property]}")
        print(f"    Parameters: {[p.parameterID for p in fluid.parameter]}")
        print(f"    Measurements: {len(fluid.measurement)}")
        
        # Show first measurement details
        if fluid.measurement:
            first_meas = fluid.measurement[0]
            print(f"    First measurement:")
            print(f"      Property value: {first_meas.propertyValue[0].propValue if first_meas.propertyValue else 'N/A'}")
            print(f"      Parameter values: {len(first_meas.parameterValue)} parameters")
            for pv in first_meas.parameterValue:
                print(f"        - {pv.parameterID}: {pv.paramValue}")
else:
    print("No documents were created.")


Document 1 summary:
  - Compounds: ['L-Histidine', 'Oxalic Acid']
  - Number of fluids: 2

  Fluid 1:
    Compounds: ['L-Histidine', 'Oxalic Acid']
    Properties: ['viscosity']
    Parameters: ['parameter_temperature', 'parameter_mole_fraction_lhistidine', 'parameter_mole_fraction_oxalicacid']
    Measurements: 7
    First measurement:
      Property value: 0.009800000000000001
      Parameter values: 3 parameters
        - parameter_temperature: 293.15
        - parameter_mole_fraction_lhistidine: 0.333
        - parameter_mole_fraction_oxalicacid: 0.667

  Fluid 2:
    Compounds: ['L-Histidine', 'Oxalic Acid']
    Properties: ['density']
    Parameters: ['parameter_temperature', 'parameter_mole_fraction_lhistidine', 'parameter_mole_fraction_oxalicacid']
    Measurements: 7
    First measurement:
      Property value: 1.0843
      Parameter values: 3 parameters
        - parameter_temperature: 293.15
        - parameter_mole_fraction_lhistidine: 0.333
        - parameter_mole_fractio

## Step 6: Verify Document Structure

Let's verify that:
1. Each document has separate fluids for viscosity and density
2. Each fluid has only one property type
3. Parameters include mole fractions and temperature


In [16]:
# Verify structure across all documents
print("Verifying structure across all documents:\n")

all_fluids_have_one_property = True
fluids_with_multiple_properties = []
total_fluids = 0
viscosity_fluids = 0
density_fluids = 0

for doc_idx, doc in enumerate(documents):
    for fluid in doc.fluid:
        total_fluids += 1
        if len(fluid.property) != 1:
            all_fluids_have_one_property = False
            fluids_with_multiple_properties.append((doc_idx, len(fluid.property)))
        
        if fluid.property:
            prop_id = fluid.property[0].propertyID
            if prop_id == "viscosity":
                viscosity_fluids += 1
            elif prop_id == "density":
                density_fluids += 1

print(f"Total documents: {len(documents)}")
print(f"Total fluids: {total_fluids}")
print(f"  - Viscosity fluids: {viscosity_fluids}")
print(f"  - Density fluids: {density_fluids}")

if all_fluids_have_one_property:
    print("\n✅ All fluids have exactly one property (as required)")
else:
    print(f"\n❌ Found {len(fluids_with_multiple_properties)} fluids with multiple properties:")
    for doc_idx, prop_count in fluids_with_multiple_properties[:5]:
        print(f"  Document {doc_idx}: {prop_count} properties")


Verifying structure across all documents:

Total documents: 229
Total fluids: 458
  - Viscosity fluids: 229
  - Density fluids: 229

✅ All fluids have exactly one property (as required)


## Step 7: Check Output Files

List the JSON files that were created in the outputs folder.


In [17]:
# List all JSON files created
json_files = list(output_dir.glob('*.json'))
print(f"Created {len(json_files)} JSON files:\n")

for json_file in sorted(json_files)[:10]:  # Show first 10
    size_kb = os.path.getsize(json_file) / 1024
    print(f"  {json_file.name}: {size_kb:.2f} KB")

if len(json_files) > 10:
    print(f"  ... and {len(json_files) - 10} more files")


Created 229 JSON files:

  1-Ethyl-3-methylimid_124-TRIAZOLE.json: 6.82 KB
  1-Ethyl-3-methylimid_1H-Benzotriazole.json: 6.85 KB
  1-Ethyl-3-methylimid_1H-Tetrazole.json: 6.78 KB
  1-Ethyl-3-methylimid_BENZIMIDAZOLE.json: 6.82 KB
  1-Ethyl-3-methylimid_SUCCINIMIDE.json: 6.79 KB
  1-Ethyl-3-methylimid_imidazole.json: 6.76 KB
  1-Ethyl-3-methylimid_phenol.json: 6.72 KB
  1-hydroxyethyl-14-di_glycerol.json: 45.37 KB
  32503-27-8_chloroacetic_acid.json: 24.19 KB
  5137-55-3_1-octanol.json: 6.78 KB
  ... and 219 more files


## Step 8: Summary Statistics

Show summary statistics about the created documents.


In [18]:
# Summary statistics
total_measurements = 0
documents_with_both_properties = 0

for doc in documents:
    has_viscosity = any(
        f.property and f.property[0].propertyID == "viscosity" 
        for f in doc.fluid
    )
    has_density = any(
        f.property and f.property[0].propertyID == "density" 
        for f in doc.fluid
    )
    
    if has_viscosity and has_density:
        documents_with_both_properties += 1
    
    for fluid in doc.fluid:
        total_measurements += len(fluid.measurement)

print("Summary Statistics:")
print(f"  Total documents: {len(documents)}")
print(f"  Documents with both viscosity and density: {documents_with_both_properties}")
print(f"  Total fluids: {total_fluids}")
print(f"  Total measurements: {total_measurements}")
print(f"  Average measurements per fluid: {total_measurements / total_fluids:.1f}" if total_fluids > 0 else "  Average measurements per fluid: 0")


Summary Statistics:
  Total documents: 229
  Documents with both viscosity and density: 229
  Total fluids: 458
  Total measurements: 4356
  Average measurements per fluid: 9.5


## Summary

This notebook demonstrates:

1. **Direct CSV Reading**: The `data_from_csv` method now reads `joined_viscosity_density.csv` format directly
2. **Compound Combination Grouping**: Each unique set of compounds (Component#1, Component#2, Component#3) creates a separate document
3. **Property Separation**: Each CSV row creates separate fluids for viscosity and density in the same document
4. **Automatic Parameter Extraction**: Mole fractions (X#1, X#2, X#3) and temperature are automatically extracted as parameters
5. **File Output**: Each compound combination is saved as a separate JSON file in the `outputs/` folder

### Key Features:
- ✅ Each compound combination gets its own JSON file
- ✅ Each fluid contains only one property type (viscosity OR density)
- ✅ Parameters include mole fractions for all compounds and temperature
- ✅ All measurements from the CSV are preserved with proper structure
